#### Classical Machine Learning for Text

- **Goal:** Establish a strong baseline. Understand how to turn text into numbers and the mechanics of classification (Weights & Gradients).
  - **Vectorization:** Loading **IMDB**. Bag of Words (`CountVectorizer`) vs. TF-IDF (`TfidfVectorizer`). Converting Sparse Matrices to PyTorch Tensors.
 
- **Sources:**
  - text-feature-extraction: https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
  - huggingface datasets: https://huggingface.co/datasets/stanfordnlp/imdb

In [1]:
# suppress warnings 
from warnings import filterwarnings
filterwarnings("ignore") 

# import the appropriate libraries
import pandas as pd
import torch
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer #text feature extraction 
from utils import get_imdb_ds, save_sklearn_features, create_dataloaders, save_tensors

# load in environment such HF_token
from dotenv import load_dotenv
load_dotenv("../../.env")

True

In [3]:
# Load and prepare dataset using utility function (now includes validation split)
train_df, val_df, test_df = get_imdb_ds(seed=204, train_size=10000, val_size=200, test_size=250)

In [4]:
# Display first few rows
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

Train size: 10000, Val size: 200, Test size: 250


In [5]:
train_df.head()

,text,label
0,"When I saw this movie, I couldn't believe my e...",0
1,LOL!!! delirious was so funny.. i was in tears...,1
2,This movie was bad from the start. The only pu...,0
3,Richard Attenborough who already given us magn...,1
4,This ranks as one of the worst movies I've see...,0


### **The "So What": Why Vectorization?**

**1. The Problem: Computers don't understand text.**
Computers and mathematical models operate on numbers (matrices, vectors), not words. You cannot calculate the gradient of the word "awesome". Furthermore, standard algorithms (Logistic Regression, Neural Networks) require inputs to have a **fixed size** (e.g., exactly 2000 inputs per example). Text is unstructured and variable in length (one review is 3 words, another is 500).

**2. The Solution: Vectorization.**
Vectorization is the process of converting variable-length text into fixed-length numeric vectors. It bridges the gap between human language and machine calculation.

**3. The Goal:** 
To create a numerical representation where the presence of specific words (features) can be mathematically weighted to predict an outcome (label).

**4. What Follows?**
Once our text is converted into tensors (vectors):
- **Day 2:** We will build a **Logistic Regression** model (a simple neural network) to learn which words (features) drive positive or negative sentiment.
- **Future:** We will see how this "Bag of Words" approach limits us (ignoring context/order) and how **Embeddings** (Word2Vec, BERT) solve that.

---

#### **Dataset Breakdown**
- **Feature:** Free form text
- **Label:** Binary, 0 is negative and 1 is positive. (Note: For multi-class classification, e.g., 20 topics, labels must be numeric integers from 0 to 19).

### **Vectorization Concepts**
The general process of turning a collection of text documents into numerical feature vectors. This is used for text feature extraction. 

#### **Vectorization Steps**
- **Tokenization:** Break text into separate tokens (e.g., using white space as a separator).
  - **Selection:** If `max_features` is set, only the top N most frequent tokens are kept.
  - **Indexing:** The selected tokens are sorted **alphabetically** and assigned indices from 0 to N-1 (e.g., "apple" is 0, "zebra" is 1999).
- **Counting:** Count occurrences of each token in a given text/document to create a frequency vector.
- **Weighting & Normalization (Optional/Method Dependent):** 
  - **Weighting (IDF):** Reduce the weight of tokens that appear frequently across the entire corpus (like "the", "is") as they are less informative.
  - **Normalization ($L_2$):** Scale the final vector so that it has a unit norm (length of 1), allowing for comparison between documents of different lengths.

#### **Popular Vectorization Concepts**
- **Bag of Words (or Bag of n-grams):** Uses **raw counts** of tokens. It generally has **no weighting** (all words are treated equally based on frequency) and **no normalization** (longer documents result in vectors with larger magnitudes).
  - **Corpus:** Represented by a matrix with one row per document and one column per token occurring in the corpus.
- **Sparsity:** Most documents only contain a very small subset of the entire corpus vocabulary. Thus, the matrix for the corpus will be very sparse (mostly zeros). The `scipy.sparse` package is used to store the matrix in a sparse representation.
- **Using Stop Words:** We often remove common words (like "the", "is") to reduce noise and dimensionality. **Caveat:** While we use `stop_words="english"` in this notebook to limit our feature space to 2000, this is not always a good idea. Words like "not" or "but" can be crucial for sentiment (e.g., "not good"). Modern deep learning models generally prefer to keep all words.
- **TF-IDF Term Weighting:** Bag of words doesn't account for document frequency. TF-IDF adds both **Weighting** (IDF) and **Normalization** ($L_2$). 
  
  $$ \text{tf-idf}(t,d) = \text{tf}(t,d) \times \text{idf}(t) $$

   - **Term Frequency (tf):** Number of times a term occurs in a given document.
   - **Inverse Document Frequency (idf):** 
     
     $$ \text{idf}(t) = \log\left(\frac{1+n}{1+\text{df}(t)}\right) + 1 $$
     
     *Note: This is Scikit-Learn's specific formula, which has two adjustments over the classic textbook version ($\log(\frac{n}{\text{df}(t)})$):*
     *   ***+1 inside the log*** *(e.g., `1+df(t)`): This is **smoothing**. It prevents division-by-zero errors.*
     *   ***+1 outside the log***: *This ensures that all IDF weights are at least 1 (i.e., non-negative), preventing any feature from being completely zeroed out by the IDF component.*
     
     Where $n$ is the number of documents in the document set and $\text{df}(t)$ is the number of documents in the document set that contain the term $t$. 
     
     TF-IDF vectors are often normalized by the Euclidean norm:
     
     $$ v_{norm} = \frac{v}{||v||_2} $$

### **Mini-Example: Bag of Words vs. TF-IDF**
Before applying this to the full IMDB dataset, let's look at a toy example to see the difference between raw counts (Bag of Words) and weighted scores (TF-IDF).

Notice how in TF-IDF:
1. Common words like "this" and "movie" (which appear in all 3 docs) have **lower scores** because they are less unique.
2. Unique words like "love", "hate", and "okay" have **higher scores**.

In [5]:
# Fake Corpus
corpus = [
    "I love this movie",
    "I hate this movie",
    "This movie is okay"
]

# 1. Bag of Words (Raw Counts)
# Note: By default, single characters like 'I' are ignored (token_pattern='(?u)\b\w\w+\b')
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(corpus)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=count_vec.get_feature_names_out(), index=["Doc 1", "Doc 2", "Doc 3"])

# 2. TF-IDF (Weighted Scores)
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(corpus)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vec.get_feature_names_out(), index=["Doc 1", "Doc 2", "Doc 3"])

print("--- Bag of Words (Raw Counts) ---")
print(bow_df)
print("\n--- TF-IDF (Weighted Scores) ---")
print(tfidf_df.round(2))

--- Bag of Words (Raw Counts) ---
       hate  is  love  movie  okay  this
Doc 1     0   0     1      1     0     1
Doc 2     1   0     0      1     0     1
Doc 3     0   1     0      1     1     1

--- TF-IDF (Weighted Scores) ---
       hate    is  love  movie  okay  this
Doc 1  0.00  0.00  0.77   0.45  0.00  0.45
Doc 2  0.77  0.00  0.00   0.45  0.00  0.45
Doc 3  0.00  0.61  0.00   0.36  0.61  0.36


In [6]:
X_train_text = train_df["text"]
y_train = train_df["label"]

X_val_text = val_df["text"]
y_val = val_df["label"]

X_test_text = test_df["text"]
y_test = test_df["label"]
max_features = 2000 # Selects the top 2000 most frequent words (after stop words removal)

## TF-IDF

In [8]:
# min_df=2: Ignore terms that appear in less than 2 documents (removes noise/typos)
# max_df=0.8: Ignore terms that appear in more than 80% of documents (removes corpus-specific stop words)
tfidf_vectorizer = TfidfVectorizer(max_features=max_features, stop_words="english", min_df=2, max_df=0.8) 
X_train_tfidf_sparse = tfidf_vectorizer.fit_transform(X_train_text)
X_val_tfidf_sparse = tfidf_vectorizer.transform(X_val_text)
X_test_tfidf_sparse = tfidf_vectorizer.transform(X_test_text)

## Save Sparse vectorization features for sklearn models
We will save the sparse features to disk so they can be loaded directly in the next session (Day 2).

In [9]:
# Assuming you have these variables from your notebook:
# X_train_tfidf_sparse, y_train, etc.

save_sklearn_features(
    X_train_tfidf_sparse, y_train,
    X_val_tfidf_sparse, y_val,
    X_test_tfidf_sparse, y_test,
    base_filename='datasets/imdb_sklearn'
)

Sparse features and labels saved with prefix 'datasets/imdb_sklearn'


### **Critical Limitations & Key Concepts**

It is important to understand the limitations of this classical approach:

1.  **Fixed Vocabulary (OOV):**
    - The vectorizer is **fixed** on the training set. If a word like "superhero" appears in the Test set but was never seen in Training, it is **ignored**.

2.  **Lack of Semantic Meaning:**
    - Words like "good" and "great" are treated as completely different features (orthogonal). The model doesn't know they are similar.

3.  **Lack of Sequence (Bag of Words):**
    - "Dog bites man" and "Man bites dog" produce the exact same vector. The order of words is lost.

4.  **Sparsity & Overfitting:**
    - The matrices are mostly zeros. High dimensionality relative to sample size can lead to **overfitting** (Curse of Dimensionality).

5.  **Polysemy:**
    - The word "bank" is the same feature whether it refers to a river or money. The model cannot distinguish meaning based on context.

6.  **Data Leakage Risk:**
    - We strictly `fit` on Train and `transform` on Test. Fitting on the whole dataset causes the model to "peek" at the test vocabulary.

## Convert to PyTorch Tensor for Dense Array

In [10]:
X_train_dense = X_train_tfidf_sparse.toarray()
X_val_dense = X_val_tfidf_sparse.toarray()
X_test_dense = X_test_tfidf_sparse.toarray()
X_train_tensor = torch.tensor(X_train_dense, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_dense, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_dense, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

In [11]:
print("\n--- Ready for Day 2 (Model Building) ---")
print(f"X_train_tensor: {X_train_tensor.shape} (Type: {X_train_tensor.dtype})")
print(f"y_train_tensor: {y_train_tensor.shape} (Type: {y_train_tensor.dtype})")
print(f"X_val_tensor: {X_val_tensor.shape} (Type: {X_val_tensor.dtype})")
print(f"y_val_tensor: {y_val_tensor.shape} (Type: {y_val_tensor.dtype})")
print(f"X_test_tensor: {X_test_tensor.shape} (Type: {X_test_tensor.dtype})")
print(f"y_test_tensor: {y_test_tensor.shape} (Type: {y_test_tensor.dtype})")


--- Ready for Day 2 (Model Building) ---
X_train_tensor: torch.Size([10000, 2000]) (Type: torch.float32)
y_train_tensor: torch.Size([10000, 1]) (Type: torch.float32)
X_val_tensor: torch.Size([200, 2000]) (Type: torch.float32)
y_val_tensor: torch.Size([200, 1]) (Type: torch.float32)
X_test_tensor: torch.Size([250, 2000]) (Type: torch.float32)
y_test_tensor: torch.Size([250, 1]) (Type: torch.float32)


In [12]:
# To see the vocabulary:
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\nRandom 10 words in vocabulary: {feature_names[1050:1060]}")


Random 10 words in vocabulary: ['line' 'lines' 'list' 'listen' 'literally' 'little' 'live' 'lived'
 'lives' 'living']


## Create PyTorch DataLoaders and Save

We will wrap the tensors into a `TensorDataset` and create a `DataLoader` for batching. We also save the processed tensors to disk so they can be loaded directly in the next session (Day 2).

In [13]:
# Create DataLoaders using utility
train_loader, val_loader, test_loader = create_dataloaders(X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor, batch_size=32)

# Save tensors for Day 2 using utility
save_tensors(X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor, 'datasets/imdb_tensors.pt')

Tensors saved to 'datasets/imdb_tensors.pt'
